# MiniLM NLI Zero-Shot Classifier

**Model:** cross-encoder/nli-MiniLM2-L6-H768 | **Size:** 71MB | **Product:** prod-eadndoekwok7u

A MiniLM cross-encoder trained on NLI (Natural Language Inference) data, enabling zero-shot text classification without task-specific fine-tuning. Classify text into any custom label set by posing classification as an NLI hypothesis-checking task.

## Use Cases
- Zero-shot topic classification without training data
- Dynamic intent detection for chatbots and virtual assistants
- News and document categorization with flexible label sets
- Rapid prototyping of classification tasks before fine-tuning

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'minilm-nli-zero-shot'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send a text sequence with candidate labels for zero-shot classification. No training data required — define any label set at inference time.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample zero-shot classification examples
examples = [
    {
        'sequences': 'The new GPU delivers exceptional performance for deep learning workloads.',
        'candidate_labels': ['technology', 'sports', 'politics', 'finance']
    },
    {
        'sequences': 'I need to cancel my subscription and get a refund for the unused months.',
        'candidate_labels': ['billing', 'technical support', 'account management', 'product feedback']
    },
    {
        'sequences': 'The quarterly earnings exceeded analyst expectations by 15 percent.',
        'candidate_labels': ['business', 'science', 'entertainment', 'health']
    },
]

for example in examples:
    payload = json.dumps({'inputs': example['sequences'], 'parameters': {'candidate_labels': example['candidate_labels']}})
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'Text: "{example["sequences"][:70]}"')
            print(f'Labels: {example["candidate_labels"]}')
            print(f'Classification: {result}
')
        except json.JSONDecodeError:
            print(f'Raw response: {result_raw}')
    except Exception as e:
        print(f'Inference failed: {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')